In [1]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

The neural network we are going to recreate is made up of only one neuron. Basically, it is just a perceptron and we are going to try to optimize that.

An Important insight is that these neural networks cannot really generate negative values here. It is a leaky abstraction. Because of the fact that we are using relu activation we are unable to generate a negative value. Because it just clips it down to a negative value. And, also the gradient is such that if the activations are negative. Then, the gradient should be zero.

In [2]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([11, 13, 19, 18, 21, -2, 0, -3, 9, -2, -1, 5])

In [1055]:
weight = np.array([1, 1, 1])
bias = np.array(1)

In [1056]:
N = len(X)
learning_rate = 0.001

In [1060]:
for i in tqdm(range(100000)):
    activations = np.maximum(0, X @ weight + bias)
    error = activations - Y

    error[activations <= 0] = 0


    gradient_w = 2/N *  np.sum( error * X.T , axis=1)
    gradient_b = 2/N *  np.sum(error)





    weight = weight - learning_rate*gradient_w
    bias = bias - learning_rate*gradient_b

    # if i % 100 == 0:
    # print(f"Weight = {weight} Bias = {bias}")

100%|██████████| 100000/100000 [00:02<00:00, 36883.38it/s]


In [1061]:
print(f"Weight = {weight} Bias = {bias}")

Weight = [ 4.00000012  1.00000004 -1.00000001] Bias = 4.9999995530323265


Another important insight is that if it jumps too far in the negative, it is hard for it to recover because the gradient then just collapses immmediately.

In [2]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([[123], [129], [243], [168], [201], [169], [357], [74], [322], [250], [202], [707]])


In [22]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

In [23]:
# for i in range(12):
#     f0 = W1 @ X[i] + B1
#     h1 = np.maximum(f0, 0)
#     print(W2 @ h1 + B2)


In [3]:
X = X.reshape(12, 3, 1)
Y = Y.reshape(12, 1, 1)

In [8]:
class NeuralNetwork():

    def __init__(self, X, Y):

        W0 = np.random.uniform(0, 0.3, size=(1, 100, 3))

        B0 = np.random.uniform(0, 0.3, size=(1, 100, 1))

        W1 = np.random.uniform(0, 0.3, size=(1, 100, 100))

        B1 = np.random.uniform(0, 0.3, size=(1, 100, 1))

        W2 = np.random.uniform(0, 0.3, size=(1, 1, 100))

        B2 = np.random.uniform(0, 0.3, size=(1, 1, 1))

        self.W = [W0, W1, W2]

        self.B = [B0, B1, B2]

        self.X = X

        self.Y = Y

        self.f = [0, 0, 0]

        self.h = [X, 0, 0]

        self.gradients_f = [0, 0, 0]

        self.gradients_W = [0, 0, 0]

        self.gradients_B = [0, 0, 0]

        # self.losses = []

    def forward_pass(self):

        for i in range(2):

            self.f[i] = self.W[i] @ self.h[i] + self.B[i]

            self.h[i+1] = np.maximum(self.f[i], 0)

        self.f[2] = self.W[2] @ self.h[2] + self.B[2]

    def get_gradients(self):

        diagonal_matrices_1 = []
        for i in range(12):
            diagonal_matrix_1 = np.diag(self.f[1][i].flatten())
            np.fill_diagonal(diagonal_matrix_1, np.maximum(diagonal_matrix_1.diagonal(), 0))

            diagonal_matrices_1.append(diagonal_matrix_1)

        diagonal_matrices_1 = np.array(diagonal_matrices_1)

        diagonal_matrices_0 = []
        for i in range(12):
            diagonal_matrix_0 = np.diag(self.f[0][i].flatten())
            np.fill_diagonal(diagonal_matrix_0, np.maximum(diagonal_matrix_0.diagonal(), 0))

            diagonal_matrices_0.append(diagonal_matrix_0)

        diagonal_matrices_0 = np.array(diagonal_matrices_0)

        self.gradients_f[2] = 2 * (self.f[2] - self.Y)
        self.gradients_f[1] = self.gradients_f[2] @ self.W[2] @ diagonal_matrices_1
        self.gradients_f[0] = self.gradients_f[1] @ self.W[1] @ diagonal_matrices_0

        for i in range(3):
            self.gradients_B[i] = np.mean(
                np.transpose(self.gradients_f[i], (0, 2, 1)), axis=0
            )[np.newaxis, :, :]

            self.gradients_W[i] = np.mean(
                np.transpose(self.h[i] @ self.gradients_f[i], (0, 2, 1)), axis=0
            )[np.newaxis, :, :]

    def gradient_descent(self, learning_rate= 0.00000001):

        for i in range(3):
            self.B[i] = self.B[i] - learning_rate*self.gradients_B[i]
            self.W[i] = self.W[i] - learning_rate*self.gradients_W[i]

    def calculate_mse(self):
        return np.mean((self.Y - self.f[2]) ** 2)

    def train(self, epochs=100, learning_rate=0.00000001):

        for _ in tqdm(range(epochs)):

            self.forward_pass()
            self.get_gradients()
            self.gradient_descent(learning_rate)

            # self.losses.append(self.calculate_mse())

In [9]:
nn = NeuralNetwork(X, Y)


In [15]:
nn.train(epochs=30000)

100%|██████████| 20000/20000 [09:11<00:00, 36.26it/s]


In [16]:
nn.calculate_mse()

7.710416196470317

In [17]:
for i in range(12):
    print(nn.f[2][i], Y[i])

[[127.6815986]] [[123]]
[[132.99691832]] [[129]]
[[243.95884288]] [[243]]
[[170.03920898]] [[168]]
[[201.76617991]] [[201]]
[[173.34638395]] [[169]]
[[361.26273822]] [[357]]
[[71.97492627]] [[74]]
[[319.31863865]] [[322]]
[[250.50226283]] [[250]]
[[202.04660494]] [[202]]
[[706.39963725]] [[707]]


In [ ]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

The truth is there is no proof of convergence for deep neural networks because backpropogation doesn't find the global minimum and is often stuck in a local minima. How would we arrange these matrices in such a way that it would find the global minima is still quite a mystery to me.

It is just super easy to get stuck in a local minima and you might not ever be able to get out of that local minima because you are modifying all of those weights at the same time.